In [1]:
# Télécharge, via l'API transport.data.gouv.fr (même routine que
# scripts/rafraichir_gtfs.py : recuperer_datasets_public_transit +
# resultat_pour_page_url, pas un simple re-téléchargement de
# ressource_url en cache — resultat_pour_page_url ré-interroge le PAN
# pour retrouver la ressource *actuelle* du dataset, potentiellement
# différente si la ressource a changé d'URL depuis le dernier
# enregistrement de provenance), tous les GTFS déjà associés à un
# dataset dans data/gtfs_sources.json — dans data/GTFS_temp/, PAS
# data/GTFS/ (ne remplace rien automatiquement, à l'écart pour
# inspection/comparaison manuelle avant de basculer si besoin).
#
# Les GTFS de gtfs_sources.json sans page_url (associés seulement pour
# leur académie/zone, cf. src.vacances_scolaires — jamais liés à un
# dataset PAN) sont ignorés : rien à télécharger pour eux ici.

import os

from src.transport_data_gouv import (
    charger_provenance,
    recuperer_datasets_public_transit,
    resultat_pour_page_url,
    telecharger_gtfs,
)

DOSSIER_TEMP = os.path.join("data", "GTFS_temp")
os.makedirs(DOSSIER_TEMP, exist_ok=True)

provenance = charger_provenance()
a_telecharger = {f: info for f, info in provenance.items() if info.get("page_url")}
print(f"{len(a_telecharger)} GTFS liés à transport.data.gouv.fr à télécharger dans {DOSSIER_TEMP}\n")

print("Récupération du catalogue transport.data.gouv.fr...")
datasets = recuperer_datasets_public_transit()

reussis, introuvables, echecs = [], [], []
for nom_fichier, info in sorted(a_telecharger.items()):
    resultat = resultat_pour_page_url(info["page_url"], info.get("ressource_url"), datasets)
    if resultat is None:
        print(f"⚠ {nom_fichier} : dataset introuvable sur transport.data.gouv.fr (page supprimée/déplacée ?)")
        introuvables.append(nom_fichier)
        continue
    try:
        contenu = telecharger_gtfs(resultat)
        chemin_cible = os.path.join(DOSSIER_TEMP, nom_fichier)
        with open(chemin_cible, "wb") as f:
            f.write(contenu)
        print(f"✓ {nom_fichier} ({len(contenu) / 1e6:.1f} Mo, màj {resultat['ressource_maj']})")
        reussis.append(nom_fichier)
    except Exception as e:
        print(f"✗ {nom_fichier} : {type(e).__name__}: {e}")
        echecs.append(nom_fichier)

print(f"\n{len(reussis)} réussi(s), {len(introuvables)} introuvable(s), {len(echecs)} échec(s)")
if introuvables:
    print("Introuvables :", introuvables)
if echecs:
    print("Échecs :", echecs)

54 GTFS liés à transport.data.gouv.fr à télécharger dans data/GTFS_temp

Récupération du catalogue transport.data.gouv.fr...


✓ Albi_libea-reseau-urbain.zip (0.0 Mo, màj 2026-06-22T08:47:55.785000Z)


✓ Ales_gtfs-is-20260704.zip (0.3 Mo, màj 2026-06-22T13:25:53.192000Z)


✓ Amiens_gtfs-fusion-20260721-1000.zip (3.0 Mo, màj 2026-07-22T09:10:44.162000Z)


✓ Angers_gtfs.zip (3.0 Mo, màj 2026-07-30T00:00:46.000000Z)


...

✓ Toulouse_tisseo_gtfs_v2_r5py.zip (18.7 Mo, màj 2026-08-11T03:51:50.265457Z)


✓ Tours_filbleu_gtfs14.zip (10.5 Mo, màj 2026-08-08T06:18:14.925161Z)


✓ Valence_gtfs-citea-20260517-20260628.zip (11.5 Mo, màj 2026-05-18T07:14:00.640000Z)


✓ Valenciennes_google-transit.zip (1.3 Mo, màj 2026-07-23T10:12:16.530000Z)


✓ Vannes_GTFS.zip (8.3 Mo, màj 2026-07-28T15:22:21.439000Z)

53 réussi(s), 0 introuvable(s), 1 échec(s)
Échecs : ['Lyon_GTFS_TCL.zip']
